# Proyek Akhir MLOps: Sistem Machine Learning Prediksi Diabetes

- **Nama:** robil
- **Username Dicoding:** robil

Notebook ini membangun *machine learning pipeline* dengan **TensorFlow Extended (TFX)** untuk memprediksi apakah seorang pasien menderita diabetes. Seluruh komponen dijalankan dengan orchestrator **Apache Beam** (`BeamDagRunner`) dan artefaknya disimpan di folder `robil-pipeline`. Model hasil pipeline kemudian di-deploy ke **Render** menggunakan TensorFlow Serving dan dimonitor dengan **Prometheus** dan **Grafana**.

Komponen: **ExampleGen → StatisticsGen → SchemaGen → ExampleValidator → Transform → Tuner → Trainer → Resolver → Evaluator → Pusher**.

Struktur kode dibuat modular agar mudah dirawat (*clean code*):

| Berkas | Isi |
| --- | --- |
| `modules/diabetes_transform.py` | `preprocessing_fn` untuk komponen Transform |
| `modules/diabetes_trainer.py` | `tuner_fn` (Tuner) dan `run_fn` (Trainer) |
| `modules/components.py` | inisialisasi seluruh komponen TFX |
| `modules/pipeline.py` | pembuatan objek pipeline untuk Apache Beam |
| `local_pipeline.py` | *entry point* untuk menjalankan pipeline dari terminal |

## 1. Import Library

In [1]:
import glob
import json
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import pandas as pd
import tensorflow as tf
import tensorflow_data_validation as tfdv
import tensorflow_model_analysis as tfma
import tfx
from absl import logging
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner

from modules.components import ComponentConfig, init_components
from modules.pipeline import init_local_pipeline

print("TensorFlow:", tf.__version__)
print("TFX:", tfx.__version__)

2026-11-22 13:15:45.611198: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.10.1
TFX: 1.11.0


## 2. Konfigurasi Path

Nama pipeline mengikuti ketentuan `<username_dicoding>-pipeline`. Folder `serving_model_dir` adalah tujuan model yang di-*push* oleh Pusher dan nantinya dimasukkan ke image TensorFlow Serving.

In [2]:
PIPELINE_NAME = "robil-pipeline"

DATA_ROOT = "data"
TRANSFORM_MODULE_FILE = os.path.abspath("modules/diabetes_transform.py")
TRAINER_MODULE_FILE = os.path.abspath("modules/diabetes_trainer.py")

PIPELINE_ROOT = PIPELINE_NAME
METADATA_PATH = os.path.join(PIPELINE_ROOT, "metadata.sqlite")
SERVING_MODEL_DIR = "serving_model_dir"

## 3. Dataset

Dataset yang digunakan adalah **Pima Indians Diabetes Database** (Kaggle/UCI): 768 data pasien perempuan keturunan Pima berusia minimal 21 tahun dengan 8 fitur numerik dan 1 label `Outcome` (1 = diabetes, 0 = tidak diabetes).

In [3]:
df = pd.read_csv(os.path.join(DATA_ROOT, "diabetes.csv"))
print("Ukuran dataset:", df.shape)
df.head()

Ukuran dataset: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
print(df["Outcome"].value_counts())
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("\nJumlah nilai 0 (dianggap missing value):")
print((df[zero_cols] == 0).sum())

0    500
1    268
Name: Outcome, dtype: int64

Jumlah nilai 0 (dianggap missing value):
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64


## 4. Modul Transform

Modul transform mengganti nilai 0 pada `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, dan `BMI` dengan rata-rata nilai valid, lalu menstandardisasi seluruh fitur dengan *z-score*. Label diubah menjadi `int64`.

In [5]:
print(open(TRANSFORM_MODULE_FILE, encoding="utf-8").read())

"""Transform module untuk dataset Pima Indians Diabetes."""

import tensorflow as tf
import tensorflow_transform as tft

NUMERIC_FEATURES = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]

# Kolom yang nilai 0-nya berarti data hilang (secara medis tidak mungkin 0).
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

LABEL_KEY = "Outcome"


def transformed_name(key):
    """Menambahkan akhiran _xf pada nama fitur hasil transformasi."""
    return key + "_xf"


def _fill_zero_with_mean(values):
    """Mengganti nilai 0 (data hilang) dengan rata-rata nilai yang bukan 0."""
    is_valid = tf.not_equal(values, 0.0)
    valid_sum = tft.sum(tf.where(is_valid, values, tf.zeros_like(values)))
    valid_count = tft.sum(tf.cast(is_valid, tf.float32))
    mean_valid = valid_sum / tf.maximum(valid_count, 1.0)
    return tf.where(is_valid, values, tf.fill(tf.shape(value

## 5. Modul Tuner & Trainer

Model berupa *Dense Neural Network* (8 input → 1–3 hidden layer ReLU + Dropout → 1 output sigmoid). Jumlah hidden layer, unit, dropout, dan learning rate dicari otomatis oleh komponen **Tuner** (KerasTuner RandomSearch, 10 trial). Model disimpan dengan *serving signature* yang menerima `tf.Example` mentah.

In [6]:
print(open(TRAINER_MODULE_FILE, encoding="utf-8").read())

"""Tuner & Trainer module untuk klasifikasi diabetes.

Modul ini dipakai oleh komponen Tuner (fungsi ``tuner_fn``) dan Trainer
(fungsi ``run_fn``) agar arsitektur model dan input pipeline konsisten.
"""

import functools
import os
from typing import Any, Dict, NamedTuple, Text

import keras_tuner as kt
import tensorflow as tf
import tensorflow_transform as tft
from keras_tuner.engine import base_tuner
from tfx.components.trainer.fn_args_utils import FnArgs

LABEL_KEY = "Outcome"
MONITOR_METRIC = "val_binary_accuracy"

TunerFnResult = NamedTuple(
    "TunerFnResult",
    [("tuner", base_tuner.BaseTuner), ("fit_kwargs", Dict[Text, Any])],
)


def transformed_name(key):
    """Menambahkan akhiran _xf pada nama fitur hasil transformasi."""
    return key + "_xf"


def gzip_reader_fn(filenames):
    """Membaca TFRecord terkompresi GZIP."""
    return tf.data.TFRecordDataset(filenames, compression_type="GZIP")


def input_fn(file_pattern, tf_transform_output, num_epochs=None, batch_size=32):

## 6. Menyusun Komponen Pipeline

Seluruh komponen dibuat oleh fungsi `init_components` di `modules/components.py`. Evaluator menetapkan *threshold* `BinaryAccuracy` ≥ 0,6 dan model baru tidak boleh lebih buruk dari model baseline yang diambil oleh Resolver.

In [7]:
components = init_components(
    ComponentConfig(
        data_dir=DATA_ROOT,
        transform_module=TRANSFORM_MODULE_FILE,
        training_module=TRAINER_MODULE_FILE,
        training_steps=500,
        eval_steps=100,
        tuning_steps=100,
        serving_model_dir=SERVING_MODEL_DIR,
    )
)
[component.id for component in components]

['CsvExampleGen',
 'StatisticsGen',
 'SchemaGen',
 'ExampleValidator',
 'Transform',
 'Tuner',
 'Trainer',
 'Latest_blessed_model_resolver',
 'Evaluator',
 'Pusher']

## 7. Menjalankan Pipeline dengan Apache Beam

Pipeline dijalankan oleh `BeamDagRunner` (orchestrator Apache Beam, mode *in memory*). Metadata eksekusi tersimpan di `robil-pipeline/metadata.sqlite`.

In [8]:
logging.set_verbosity(logging.ERROR)

pipeline = init_local_pipeline(PIPELINE_NAME, PIPELINE_ROOT, METADATA_PATH, components)
BeamDagRunner().run(pipeline)

Trial 10 Complete [00h 00m 04s]
val_binary_accuracy: 0.7587500214576721

Best val_binary_accuracy So Far: 0.7850000262260437
Total elapsed time: 00h 00m 45s
INFO:tensorflow:Oracle triggered exit


INFO:tensorflow:Oracle triggered exit


Results summary
Results in robil-pipeline/Tuner/.system/executor_execution/7/.temp/7/diabetes_kt
Showing 10 best trials
Trial summary
Hyperparameters:
dropout: 0.5
num_hidden_layers: 3
units_0: 16
learning_rate: 0.001
units_1: 32
units_2: 48
Score: 0.7850000262260437
Trial summary
Hyperparameters:
dropout: 0.2
num_hidden_layers: 3
units_0: 48
learning_rate: 0.001
units_1: 32
units_2: 48
Score: 0.7806249856948853
Trial summary
Hyperparameters:
dropout: 0.30000000000000004
num_hidden_layers: 3
units_0: 64
learning_rate: 0.01
units_1: 80
units_2: 64
Score: 0.7796875238418579
Trial summary
Hyperparameters:
dropout: 0.2
num_hidden_layers: 2
units_0: 80
learning_rate: 0.01
units_1: 64
units_2: 96
Score: 0.7749999761581421
Trial summary
Hyperparameters:
dropout: 0.30000000000000004
num_hidden_layers: 3
units_0: 64
learning_rate: 0.001
units_1: 32
units_2: 16
Score: 0.7621874809265137
Trial summary
Hyperparameters:
dropout: 0.2
num_hidden_layers: 2
units_0: 128
learning_rate: 0.01
units_1: 16



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


                                                                  'Insulin_xf[0][0]',             
                                                                  'Pregnancies_xf[0][0]',         
                                                                  'SkinThickness_xf[0][0]']       
                                                                                                  
 dense_4 (Dense)                (None, 16)           144         ['concatenate_1[0][0]']          
                                                                                                  
 dropout_3 (Dropout)            (None, 16)           0           ['dense_4[0][0]']                
                                                                                                  
 dense_5 (Dense)                (None, 32)           544         ['dropout_3[0][0]']              
                                                                                                  
 dropout_4

INFO:tensorflow:struct2tensor is not available.


... (4 outputs left out)

INFO:tensorflow:Assets written to: robil-pipeline/Trainer/model/8/Format-Serving/assets


INFO:tensorflow:Assets written to: robil-pipeline/Trainer/model/8/Format-Serving/assets


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


In [9]:
sorted(os.listdir(PIPELINE_ROOT))

['CsvExampleGen',
 'Evaluator',
 'ExampleValidator',
 'Pusher',
 'SchemaGen',
 'StatisticsGen',
 'Trainer',
 'Transform',
 'Tuner',
 '_wheels',
 'metadata.sqlite']

## 8. Memeriksa Hasil Setiap Komponen

Fungsi bantu berikut mengambil artefak terbaru dari setiap komponen di folder `robil-pipeline`.

In [10]:
def latest_artifact(component, output):
    """Mengembalikan path artefak terbaru dari sebuah komponen."""
    runs = glob.glob(os.path.join(PIPELINE_ROOT, component, output, "*"))
    return max(runs, key=lambda path: int(os.path.basename(path)))

### 8.1 StatisticsGen & SchemaGen

In [11]:
stats = tfdv.load_stats_binary(
    os.path.join(latest_artifact("StatisticsGen", "statistics"), "Split-train", "FeatureStats.pb")
)
rows = []
for feature in stats.datasets[0].features:
    num = feature.num_stats
    rows.append({
        "fitur": feature.path.step[0],
        "mean": round(num.mean, 3),
        "std": round(num.std_dev, 3),
        "min": num.min,
        "max": num.max,
        "jumlah_nol": num.num_zeros,
    })
pd.DataFrame(rows)

,fitur,mean,std,min,max,jumlah_nol
0,Age,33.362,11.767,21.000,72.000000,0
1,BMI,31.911,7.912,0.000,67.099998,9
2,BloodPressure,69.594,18.377,0.000,122.000000,22
3,DiabetesPedigreeFunction,0.476,0.329,0.078,2.420000,0
4,Glucose,120.352,31.873,0.000,199.000000,4
5,Insulin,77.797,105.981,0.000,744.000000,283
6,Outcome,0.338,0.473,0.000,1.000000,388
7,Pregnancies,3.862,3.367,0.000,17.000000,84
8,SkinThickness,20.558,15.566,0.000,63.000000,168


In [12]:
schema = tfdv.load_schema_text(os.path.join(latest_artifact("SchemaGen", "schema"), "schema.pbtxt"))
tfdv.display_schema(schema)

,Type,Presence,Valency,Domain
Feature name,,,,
'Age',INT,required,,-
'BMI',FLOAT,required,,-
'BloodPressure',INT,required,,-
'DiabetesPedigreeFunction',FLOAT,required,,-
'Glucose',INT,required,,-
'Insulin',INT,required,,-
'Outcome',INT,required,,-
'Pregnancies',INT,required,,-
'SkinThickness',INT,required,,-


### 8.2 ExampleValidator

In [13]:
from tensorflow_metadata.proto.v0 import anomalies_pb2

anomalies_path = os.path.join(
    latest_artifact("ExampleValidator", "anomalies"), "Split-eval", "SchemaDiff.pb"
)
anomalies = anomalies_pb2.Anomalies()
with open(anomalies_path, "rb") as f:
    anomalies.ParseFromString(f.read())
print("Jumlah fitur dengan anomali:", len(anomalies.anomaly_info))
tfdv.display_anomalies(anomalies)

Jumlah fitur dengan anomali: 0


### 8.3 Tuner — Hyperparameter Terbaik

In [14]:
with open(os.path.join(latest_artifact("Tuner", "best_hyperparameters"), "best_hyperparameters.txt")) as f:
    best_hp = json.load(f)["values"]
best_hp

{'dropout': 0.5,
 'num_hidden_layers': 3,
 'units_0': 16,
 'learning_rate': 0.001,
 'units_1': 32,
 'units_2': 48}

### 8.4 Evaluator

In [15]:
eval_result = tfma.load_eval_result(latest_artifact("Evaluator", "evaluation"))
metrics = eval_result.slicing_metrics[0][1][""][""]
pd.DataFrame(
    {name: [value.get("doubleValue")] for name, value in sorted(metrics.items())},
    index=["nilai"],
).T

,nilai
auc,0.858992
binary_accuracy,0.769231
example_count,182.000000
false_negatives,22.000000
false_positives,20.000000
loss,0.461649
precision,0.705882
recall,0.685714
true_negatives,92.000000
true_positives,48.000000


In [16]:
blessing = os.listdir(latest_artifact("Evaluator", "blessing"))
print("Status model:", blessing)

Status model: ['BLESSED']


### 8.5 Pusher

Model yang *blessed* disalin ke `serving_model_dir`. Folder ini dimasukkan ke image TensorFlow Serving (lihat `Dockerfile`) dan di-deploy ke Render.

In [17]:
for root, dirs, files in os.walk(SERVING_MODEL_DIR):
    if root.count(os.sep) <= 1:
        print(root, dirs)

serving_model_dir ['1795353470']
serving_model_dir/1795353470 ['assets', 'variables']


## 9. Kesimpulan

Pipeline TFX berhasil dijalankan dengan Apache Beam, mulai dari *data ingestion*, validasi data, *preprocessing*, *hyperparameter tuning*, pelatihan, evaluasi, hingga *push* model. Model kemudian di-deploy ke Render dengan TensorFlow Serving, diuji melalui `robil-testing.ipynb`, dan dimonitor dengan Prometheus serta Grafana (folder `monitoring`).